# 63 - Cross-Dataset Evaluation: Secondary → Primer Test

**Skema 2:** Train di dataset sekunder, evaluasi di data test Primer.

| Train | Test | Catatan |
|-------|------|---------|
| CK+ (636/654) | Primer test (929) | Subject-wise 90/10 train/val split |
| JAFFE (213) | Primer test | Subject-wise 90/10 |
| RAF-DB (11,565) | Primer test | Official train, 90/10 untuk val |
| KDEF (2,630) | Primer test | Train set KDEF (sudah ada val terpisah) |

**Model:** Semua model (CNN, FCNN, Intermediate, CNN_TL, Intermediate_TL, Late_Fusion).
**Metrik:** Macro/Micro/Weighted F1.
**Catatan label:** Semua dataset pakai mapping 7-kelas yang sama → aman untuk cross-evaluate.

In [ ]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 32
EPOCHS = 40
PATIENCE = 12
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'benchmark' / 'crossdataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'

MODELS = [
    ('CNN', EmotionCNN, 'cnn', 0.0001),
    ('FCNN', EmotionFCNN, 'fcnn', 0.0001),
    ('Intermediate', IntermediateFusion, 'fusion', 0.0001),
    ('CNN_TL', EmotionCNNTransfer, 'cnn', 0.00005),
    ('Intermediate_TL', IntermediateFusionTransfer, 'fusion', 0.00005),
]

EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']
REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

In [ ]:
# ── Helpers ──

def make_loader(images, landmarks, labels, model_type, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == 'cnn': ds = TensorDataset(img_t, y_t)
    elif model_type == 'fcnn': ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


def load_primer_test(num_classes):
    te_img = np.load(PRIMER_DIR / 'X_test_images.npy')
    te_lm = np.load(PRIMER_DIR / 'X_test_landmarks.npy')
    y = np.load(PRIMER_DIR / 'y_test.npy')
    if num_classes == 4:
        y = REMAP_4[y]
    return te_img, te_lm, y


def load_secondary(dataset_name, num_classes):
    """Return tr_img, tr_lm, tr_y, v_img, v_lm, v_y (all 7 or 4 class)."""
    if dataset_name == 'ckplus':
        sub = f'ckplus_{num_classes}class' if num_classes == 7 else 'ckplus_4class_contempt'
        d = BENCHMARK_DIR / sub
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subs = np.load(d / 'subjects.npy', allow_pickle=True)
        # Subject-wise 90/10
        uniq = sorted(set(subs.tolist()))
        rng = np.random.RandomState(42); rng.shuffle(uniq)
        n_tr = int(len(uniq) * 0.9)
        tr_subs = set(uniq[:n_tr]); v_subs = set(uniq[n_tr:])
        tr_idx = np.where(np.isin(subs, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subs, list(v_subs)))[0]
        return X[tr_idx], L[tr_idx], y[tr_idx], X[v_idx], L[v_idx], y[v_idx]

    if dataset_name == 'jaffe':
        d = BENCHMARK_DIR / f'jaffe_{num_classes}class'
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subs = np.load(d / 'subjects.npy', allow_pickle=True)
        uniq = sorted(set(subs.tolist()))
        rng = np.random.RandomState(42); rng.shuffle(uniq)
        n_tr = int(len(uniq) * 0.9)
        tr_subs = set(uniq[:n_tr]); v_subs = set(uniq[n_tr:])
        tr_idx = np.where(np.isin(subs, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subs, list(v_subs)))[0]
        return X[tr_idx], L[tr_idx], y[tr_idx], X[v_idx], L[v_idx], y[v_idx]

    if dataset_name == 'rafdb':
        d = BENCHMARK_DIR / f'rafdb_{num_classes}class'
        X = np.load(d / 'X_train_images.npy')
        L = np.load(d / 'X_train_landmarks.npy')
        y = np.load(d / 'y_train.npy')
        tr_idx, v_idx = train_test_split(np.arange(len(y)), test_size=0.1, stratify=y, random_state=42)
        return X[tr_idx], L[tr_idx], y[tr_idx], X[v_idx], L[v_idx], y[v_idx]

    if dataset_name == 'kdef':
        d = BENCHMARK_DIR / f'kdef_{num_classes}class'
        tr_X = np.load(d / 'X_train_images.npy')
        tr_L = np.load(d / 'X_train_landmarks.npy')
        tr_y = np.load(d / 'y_train.npy')
        v_X = np.load(d / 'X_val_images.npy')
        v_L = np.load(d / 'X_val_landmarks.npy')
        v_y = np.load(d / 'y_val.npy')
        return tr_X, tr_L, tr_y, v_X, v_L, v_y

    raise ValueError(f'Unknown dataset {dataset_name}')


def train_and_eval(ModelClass, model_type, lr,
                   tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                   te_img, te_lm, te_y, emotions, save_dir):
    tr_loader = make_loader(tr_img, tr_lm, tr_y, model_type, BATCH_SIZE)
    val_loader = make_loader(v_img, v_lm, v_y, model_type, BATCH_SIZE, False)
    test_loader = make_loader(te_img, te_lm, te_y, model_type, BATCH_SIZE, False)
    model = ModelClass(num_classes=len(emotions)).to(device)
    save_path = str(save_dir / 'model.pth')
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_loader, val_loader, crit, opt, sch, device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, crit, device, model_type, emotions)
    return {'accuracy': float(r['test_accuracy']),
            'macro_f1': float(r['test_macro_f1']),
            'micro_f1': float(r['test_micro_f1']),
            'weighted_f1': float(r['test_weighted_f1'])}


def late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                     te_img, te_lm, te_y, num_classes, save_dir):
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    tr_cnn = make_loader(tr_img, tr_lm, tr_y, 'cnn', BATCH_SIZE)
    val_cnn = make_loader(v_img, v_lm, v_y, 'cnn', BATCH_SIZE, False)
    opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(cnn, tr_cnn, val_cnn, nn.CrossEntropyLoss(), opt, sch,
                device, 'cnn', EPOCHS, PATIENCE, str(save_dir / 'cnn.pth'))
    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    tr_fcnn = make_loader(tr_img, tr_lm, tr_y, 'fcnn', BATCH_SIZE)
    val_fcnn = make_loader(v_img, v_lm, v_y, 'fcnn', BATCH_SIZE, False)
    opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
    sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(fcnn, tr_fcnn, val_fcnn, nn.CrossEntropyLoss(), opt2, sch2,
                device, 'fcnn', EPOCHS, PATIENCE, str(save_dir / 'fcnn.pth'))
    cnn.load_state_dict(torch.load(save_dir / 'cnn.pth', map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(save_dir / 'fcnn.pth', map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()
    v_img_t = torch.from_numpy(v_img).permute(0, 3, 1, 2).to(device)
    v_lm_t = torch.from_numpy(v_lm).to(device)
    with torch.no_grad():
        vc = torch.softmax(cnn(v_img_t), dim=1).cpu().numpy()
        vf = torch.softmax(fcnn(v_lm_t), dim=1).cpu().numpy()
    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * vc + (1-w) * vf).argmax(axis=1)
        f = f1_score(v_y, pr, average='macro', zero_division=0)
        if f > best_wf1: best_wf1 = f; best_w = w
    print(f'    Late-fusion best w(cnn)={best_w:.2f}')
    te_img_t = torch.from_numpy(te_img).permute(0, 3, 1, 2).to(device)
    te_lm_t = torch.from_numpy(te_lm).to(device)
    with torch.no_grad():
        cp = torch.softmax(cnn(te_img_t), dim=1).cpu().numpy()
        fp = torch.softmax(fcnn(te_lm_t), dim=1).cpu().numpy()
    preds = (best_w * cp + (1-best_w) * fp).argmax(axis=1)
    m = metrics_triple(te_y, preds)
    m['best_cnn_weight'] = float(best_w)
    return m


def run_cross(dataset_name, num_classes):
    emotions = EMOTIONS_7 if num_classes == 7 else EMOTIONS_4
    print(f"\n{'='*70}")
    print(f'  CROSS: train={dataset_name} ({num_classes}c) -> test=Primer')
    print(f"{'='*70}")
    tr_img, tr_lm, tr_y, v_img, v_lm, v_y = load_secondary(dataset_name, num_classes)
    te_img, te_lm, te_y = load_primer_test(num_classes)
    print(f'  Train={len(tr_y)}  Val={len(v_y)}  PrimerTest={len(te_y)}')
    results = {}
    for model_name, ModelClass, model_type, lr in MODELS + [('Late_Fusion', None, 'late', 0.0001)]:
        key = f'{model_name}_B1'
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{dataset_name}_{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)
        if model_type == 'late':
            r = late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                  te_img, te_lm, te_y, num_classes, save_dir)
        else:
            r = train_and_eval(ModelClass, model_type, lr,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                te_img, te_lm, te_y, emotions, save_dir)
        results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")
    save_path = OUTPUT_DIR / f'cross_{dataset_name}_{num_classes}c.json'
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return results

print('Helper functions ready.')

## Run Cross-Dataset Experiments

Jalankan per-dataset untuk kontrol waktu (tiap dataset bisa 1-3 jam).

In [ ]:
all_cross = {}
# CK+
all_cross['ckplus_7c'] = run_cross('ckplus', 7)
all_cross['ckplus_4c'] = run_cross('ckplus', 4)

In [ ]:
# JAFFE
all_cross['jaffe_7c'] = run_cross('jaffe', 7)
all_cross['jaffe_4c'] = run_cross('jaffe', 4)

In [ ]:
# RAF-DB (paling besar, ~1-3 jam)
all_cross['rafdb_7c'] = run_cross('rafdb', 7)
all_cross['rafdb_4c'] = run_cross('rafdb', 4)

In [ ]:
# KDEF
all_cross['kdef_7c'] = run_cross('kdef', 7)
all_cross['kdef_4c'] = run_cross('kdef', 4)

## Ringkasan Semua Cross-Dataset Results

In [ ]:
for key, res in all_cross.items():
    print(f"\n{'='*78}")
    print(f'  {key.upper()} -> Primer Test (sorted by Macro F1)')
    print(f"{'='*78}")
    print(f"  {'Model':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for k in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[k]
        print(f"  {k:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")

# Save all
with open(OUTPUT_DIR / 'all_cross_results.json', 'w') as f:
    json.dump(all_cross, f, indent=2)
print(f'\nAll cross-dataset results saved: {OUTPUT_DIR / "all_cross_results.json"}')